# Detecting deepfakes by latent-space outlier detection

**Setup.** We take one real video and **swap a different identity into only a short
segment of frames** — so most of the clip is genuine and a small part is fake. The
task is then a classic **outlier-detection** problem: encode every frame into a
latent identity space and **find the rare anomalous frames** (the tampered ones).

## Pipeline

1. Take a **target video** of a real face (identity *A*).
2. Use **`inswapper`** from [insightface](https://github.com/deepinsight/insightface)
   to paste a **different identity** (*B*) into a **chosen range of frames** → a
   single `manipulated.mp4` that is mostly genuine with a fake segment.
3. Encode every frame with **ArcFace** into a **512-D** latent space
   (`normed_embedding`, unit length).
4. Fit a **robust model of "normal"** on the embeddings (the genuine majority) and
   score every frame; the swapped frames stand out as **outliers**.
5. Render a **playable annotated video** with face boxes + keypoints + the verdict.

> **Why this works.** ArcFace maps a face to an *identity* vector and is trained so
> that one person occupies a tight region of the sphere. The genuine frames form a
> dense cluster (always person *A*); the swapped frames jump to person *B*'s region
> and become a sparse, distant minority — i.e. **outliers**. Because they are the
> minority, we don't need any external clean reference: a robust estimator fit on all
> frames treats the genuine cluster as "normal" and flags the rest.

## Models we use (the `buffalo_l` pack)
- **SCRFD** – face detection (box + 5 keypoints),
- **ArcFace** (IResNet-50) – the 512-D identity embedding = our latent space,
- **`inswapper_128`** – the identity swap that creates the fake segment.

> ⚠️ **License / ethics.** insightface *code* is MIT, but the pretrained models and
> `inswapper` are *"non-commercial research only"*. This is an educational/research
> experiment. Only swap faces on material you have the rights and consent to use.


## 1. Setup and configuration

On macOS everything runs on CPU through `onnxruntime`. Uncomment the install line if
needed. We point `SSL_CERT_FILE` at the `certifi` bundle — the python.org build of
Python on macOS ships without root certificates, so model/asset downloads would
otherwise fail with `CERTIFICATE_VERIFY_FAILED`.

In [ ]:
# !pip install insightface onnxruntime opencv-python numpy scikit-learn matplotlib imageio-ffmpeg tqdm certifi huggingface_hub umap-learn
%matplotlib inline

import os, time, pathlib, ssl, shutil, urllib.request
import certifi
os.environ["SSL_CERT_FILE"] = certifi.where()      # fix macOS python.org SSL certs
os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()
SSL_CTX = ssl.create_default_context(cafile=certifi.where())

import numpy as np
import cv2
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from IPython.display import Video, display

SEED = 42
np.random.seed(SEED)

CONFIG = dict(
    data_dir   = pathlib.Path("data"),
    out_dir    = pathlib.Path("outputs"),
    # stable public sample videos (intel-iot-devkit)
    target_url = "https://github.com/intel-iot-devkit/sample-videos/raw/master/head-pose-face-detection-female.mp4",
    source_url = "https://github.com/intel-iot-devkit/sample-videos/raw/master/head-pose-face-detection-male.mp4",
    providers  = ["CPUExecutionProvider"],   # on Apple Silicon you may try "CoreMLExecutionProvider"
    det_size   = (640, 640),
    fps        = 15,
    stride     = 2,        # process every 2nd frame (speed)
    max_frames = 220,      # cap on processed frames
    swap_lo    = 0.45,     # swap the segment [swap_lo, swap_hi) of the processed frames
    swap_hi    = 0.68,     # -> ~23% of frames are fake, the rest genuine
)
CONFIG["data_dir"].mkdir(parents=True, exist_ok=True)
CONFIG["out_dir"].mkdir(parents=True, exist_ok=True)
print("Configuration ready.")

## 2. Helper functions

A cached downloader (with retries), a frame sampler, largest-face picker, and a video
writer that prefers **H.264 (`avc1`)** so the result plays in QuickTime / VLC / a
browser (falling back to `mp4v`).

In [ ]:
def download(url, dst, min_bytes=1000, retries=4):
    """Download url to dst unless already cached. certifi SSL context + retries."""
    dst = pathlib.Path(dst)
    if dst.exists() and dst.stat().st_size > min_bytes:
        print(f"  cache: {dst.name} ({dst.stat().st_size/1e6:.1f} MB)"); return dst
    dst.parent.mkdir(parents=True, exist_ok=True)
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    for attempt in range(1, retries+1):
        try:
            print(f"  downloading {dst.name} (try {attempt}/{retries}) ...")
            t = time.time()
            with urllib.request.urlopen(req, context=SSL_CTX) as r, open(dst, "wb") as f:
                shutil.copyfileobj(r, f)
            print(f"    {dst.stat().st_size/1e6:.1f} MB in {time.time()-t:.0f}s"); return dst
        except Exception as e:
            print(f"    failed: {e}"); time.sleep(2*attempt)
    raise RuntimeError(f"could not download {url}")

def sample_frames(path, stride=1, max_frames=None):
    """Return (indices, frames_bgr) - subsampled frames of a video."""
    cap = cv2.VideoCapture(str(path)); idxs, frames, i = [], [], 0
    while True:
        ok, fr = cap.read()
        if not ok: break
        if i % stride == 0:
            idxs.append(i); frames.append(fr)
            if max_frames and len(frames) >= max_frames: break
        i += 1
    cap.release(); return idxs, frames

def largest_face(faces):
    """Pick the largest detected face (by bbox area)."""
    if not faces: return None
    return max(faces, key=lambda f: (f.bbox[2]-f.bbox[0])*(f.bbox[3]-f.bbox[1]))

def write_video(path, frames, fps=15):
    """Write frames to an .mp4, preferring H.264 (avc1), falling back to mp4v."""
    h, w = frames[0].shape[:2]
    for cc in ("avc1", "mp4v"):
        vw = cv2.VideoWriter(str(path), cv2.VideoWriter_fourcc(*cc), fps, (w, h))
        if vw.isOpened():
            for fr in frames: vw.write(fr)
            vw.release(); print(f"  wrote {path} ({cc})"); return path
        vw.release()
    raise RuntimeError("no working video codec")

def show_bgr(img, title="", ax=None):
    ax = ax or plt.gca()
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); ax.set_title(title); ax.axis("off")

print("Helpers loaded.")

## 3. Download the material

- **target video** (`target.mp4`) — real person *A* (female); the clip we partly tamper with,
- **source video** (`source.mp4`) — one frame is **identity *B*** (male) to paste into the swapped segment.

In [ ]:
dd = CONFIG["data_dir"]
target_path = download(CONFIG["target_url"], dd/"target.mp4")
source_path = download(CONFIG["source_url"], dd/"source.mp4")

_, t_preview = sample_frames(target_path, stride=30, max_frames=1)
_, s_preview = sample_frames(source_path, stride=30, max_frames=1)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
show_bgr(t_preview[0], "target video (identity A)", axes[0])
show_bgr(s_preview[0], "source of identity B", axes[1])
plt.tight_layout(); plt.show()

## 4. The encoder: how a face becomes a 512-D vector

The detector lives or dies by the **encoder**, so it is worth understanding. In
insightface the encoder is a small **pipeline**, and the recognition model in
`buffalo_l` is **`w600k_r50`** — an **IResNet-50** trained with the **ArcFace** loss
on WebFace600K.

```
 raw frame
    │
    ▼
┌────────────────────────────┐
│ SCRFD detector             │  face bbox + 5 keypoints
│ (eyes, nose, mouth corners)│
└────────────────────────────┘
    │  similarity transform (rotate/scale) to a canonical layout
    ▼
┌────────────────────────────┐
│ aligned crop 112×112×3     │  pixels scaled to ~[-1, 1]
└────────────────────────────┘
    │
    ▼
┌──────────────────────────────────────────────┐
│ IResNet-50 backbone (the encoder)            │
│  stem: 3×3 conv, stride 1 (keeps resolution  │
│        for small face crops)                 │
│  4 stages of IR residual blocks: [3,4,14,3]  │
│   channels 64 → 128 → 256 → 512              │
│   IR block: BN─Conv3×3─BN─PReLU─Conv3×3─BN    │
│             (+ shortcut)                      │
└──────────────────────────────────────────────┘
    │  7×7×512 feature map
    ▼
┌──────────────────────────────────────────────┐
│ output head:  BN ─ Dropout ─ FC(512) ─ BN     │
└──────────────────────────────────────────────┘
    │
    ▼
┌────────────────────────────┐
│ L2-normalize → unit 512-D   │  `face.normed_embedding`  (‖v‖ = 1)
│ vector on the hypersphere   │
└────────────────────────────┘
```

**Why these choices matter for detection:**

- **Alignment first.** SCRFD's 5 landmarks drive a similarity transform to a 112×112
  crop, so pose/scale are factored out and the embedding describes **identity**, not framing.
- **IR residual blocks + PReLU.** "Improved ResNet" uses a BN-first residual block and
  `PReLU`; the 3×3 stride-1 stem avoids the aggressive early downsampling of vanilla
  ResNet, which matters for tiny 112-px faces.
- **Output head** `BN ─ Dropout ─ FC(512) ─ BN` (the ArcFace "option-E" head) yields a fixed **512-D** descriptor.
- **L2 normalization → unit hypersphere.** Every embedding has length 1, so the natural
  distance is **angular / cosine** — exactly what our outlier scores use.

**Why same-identity frames cluster — the ArcFace loss.** Training takes the angle θ
between an embedding and its class weight, adds a margin *m*, and scales by *s*:

$$ \text{logit} = s \cdot \cos(\theta_{y} + m). $$

Penalizing the angle *plus a margin* forces same-identity vectors into a **tight
angular cluster** and pushes different identities apart. So the genuine frames of
person *A* form one compact cluster, and the swapped segment (person *B*) lands
**outside** it — the gap our detector measures.

> `inswapper` itself takes the **source ArcFace embedding** as the identity condition
> and re-renders the target face region with that identity — which is precisely why a
> swapped crop encodes to a *different* point in this same latent space.


## 5. Load the insightface models

`FaceAnalysis` (`buffalo_l`) bundles **detection + landmarks + the ArcFace encoder**.
We fetch `inswapper_128.onnx` into `~/.insightface/models/` (the official auto-download
was removed; we use a mirror via `huggingface_hub`, falling back to a direct URL).

In [ ]:
from insightface.app import FaceAnalysis
from insightface import model_zoo

app = FaceAnalysis(name="buffalo_l", providers=CONFIG["providers"])
app.prepare(ctx_id=0, det_size=CONFIG["det_size"])

inswap_path = pathlib.Path.home()/".insightface"/"models"/"inswapper_128.onnx"
if not (inswap_path.exists() and inswap_path.stat().st_size > 1e8):
    try:
        from huggingface_hub import hf_hub_download
        p = hf_hub_download(repo_id="ezioruan/inswapper_128.onnx", filename="inswapper_128.onnx")
        shutil.copy(p, inswap_path)
    except Exception as e:
        print("hf_hub failed, trying direct URL:", e)
        download("https://huggingface.co/datasets/Gourieff/ReActor/resolve/main/models/inswapper_128.onnx",
                 inswap_path, min_bytes=1e8)
assert inswap_path.exists() and inswap_path.stat().st_size > 1e8, \
    "Could not fetch inswapper - download inswapper_128.onnx manually into ~/.insightface/models/"
swapper = model_zoo.get_model(str(inswap_path), providers=CONFIG["providers"])
print("Models ready:", inswap_path.name)

## 6. Build the partially-manipulated video

We take **one source face** (identity *B*) and swap it into **only** the frames in
`[swap_lo, swap_hi)`. Everything else is the untouched original — a single clip that is
mostly genuine with a fake segment, plus a per-frame **ground-truth label** `y_true`
(1 = swapped) to score against later.

In [ ]:
# 1) source face (identity B)
_, s_frames = sample_frames(source_path, stride=5, max_frames=60)
source_face = None
for fr in s_frames:
    source_face = largest_face(app.get(fr))
    if source_face is not None: break
assert source_face is not None, "no face found in the source video"
print("Source identity B ready (embedding dim =", source_face.normed_embedding.shape[0], ")")

# 2) sampled frames of the target video
frame_idx, frames = sample_frames(target_path, stride=CONFIG["stride"], max_frames=CONFIG["max_frames"])
N = len(frames)
lo, hi = int(CONFIG["swap_lo"]*N), int(CONFIG["swap_hi"]*N)
print(f"{N} frames; swapping frames [{lo}, {hi}) -> {hi-lo} fake ({(hi-lo)/N:.0%})")

# 3) swap identity B into ONLY the chosen segment
y_true, swap_fail = np.zeros(N, dtype=int), 0
manip = []
for i, fr in enumerate(tqdm(frames, desc="building")):
    if lo <= i < hi:
        faces = app.get(fr); out = fr.copy()
        if faces:
            for f in faces: out = swapper.get(out, f, source_face, paste_back=True)
            y_true[i] = 1                 # successfully swapped -> fake
        else:
            swap_fail += 1                # no face to swap -> stays genuine
        manip.append(out)
    else:
        manip.append(fr)                  # genuine frame, untouched
print(f"Fake frames: {int(y_true.sum())} | swap skipped (no face): {swap_fail}")

# 4) genuine vs swapped preview
gi = lo//2; fi = (lo+hi)//2
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
show_bgr(manip[gi], f"genuine frame #{gi} (identity A)", axes[0])
show_bgr(manip[fi], f"swapped frame #{fi} (identity B)", axes[1])
plt.tight_layout(); plt.show()

In [ ]:
# save the manipulated video (H.264) and show an inline player
manip_path = write_video(CONFIG["out_dir"]/"manipulated.mp4", manip, fps=CONFIG["fps"])
display(Video(str(manip_path), embed=False, width=480))

## 7. Encode every frame into the latent space (ArcFace 512-D)

Each frame → largest face → `normed_embedding` (512-D, $\lVert v\rVert = 1$). We also
keep each face's **box + 5 keypoints** for the annotated video later. Frames with no
detected face are masked out.

In [ ]:
def embed_frames(frames, app):
    """Return (E [N,512] with NaN where no face, mask [N], info: list of (bbox,kps) or None)."""
    embs, mask, info = [], [], []
    for fr in tqdm(frames, desc="embedding"):
        f = largest_face(app.get(fr))
        if f is None:
            embs.append(np.full(512, np.nan, np.float32)); mask.append(False); info.append(None)
        else:
            embs.append(f.normed_embedding.astype(np.float32)); mask.append(True)
            info.append((f.bbox.copy(), f.kps.copy()))
    return np.vstack(embs), np.array(mask), info

E, mask, info = embed_frames(manip, app)
X   = E[mask]                 # embeddings of frames with a detected face
y   = y_true[mask]            # their ground-truth labels (1 = swapped)
pos = np.where(mask)[0]       # original frame positions (for plotting / annotation)
print("embeddings:", E.shape, "| with face:", mask.sum(), "| fake among them:", int(y.sum()))

np.savez(CONFIG["out_dir"]/"embeddings.npz", E=E, mask=mask, y_true=y_true)
print("Saved:", CONFIG["out_dir"]/"embeddings.npz")

## 8. UMAP 2D projection of the embeddings (see the clusters)

Right after computing the embeddings, project the 512-D vectors to **2-D with UMAP**
(cosine metric, since ArcFace embeddings live on a unit sphere) and colour by the true
label. The genuine frames (one identity) should collapse into one tight cluster and the
swapped frames into a separate one — a direct visual check that the swap is separable in
the latent space, before we even pick a detector.

In [ ]:
import sys
# umap's __init__ tries to import ParametricUMAP -> tensorflow; the tensorflow build
# here is numpy-2 incompatible. Block it so umap catches the ImportError and skips it.
sys.modules.setdefault("tensorflow", None)
from umap import UMAP

emb2d = UMAP(n_components=2, n_neighbors=15, min_dist=0.1,
             metric="cosine", random_state=SEED).fit_transform(X)

plt.figure(figsize=(7, 6))
plt.scatter(emb2d[y==0, 0], emb2d[y==0, 1], s=22, c="tab:green", alpha=.8, label="real")
plt.scatter(emb2d[y==1, 0], emb2d[y==1, 1], s=22, c="tab:red",   alpha=.8, label="fake")
plt.title("UMAP 2D projection of ArcFace embeddings")
plt.xlabel("UMAP-1"); plt.ylabel("UMAP-2"); plt.legend()
plt.tight_layout(); plt.show()

## 9. Detect outliers: cosine distance to the centroid + basic thresholding

We keep it deliberately simple — **one** signal, **one** threshold:

1. **Centroid.** Take the mean of all embeddings as the centre of the *genuine*
   identity. The swapped frames are a minority, so the mean still sits in the genuine
   cluster; we run a few **trimming** iterations (drop the farthest 30%, recompute) so
   the centroid locks onto the majority and isn't dragged by the fakes.
2. **Distance.** Each frame's anomaly score is the **cosine distance to the centroid**,
   $d = 1-\cos(x,\bar{x})$. Genuine frames sit near 0; swapped frames (a different
   identity) are pushed toward 1.
3. **Threshold.** The distance histogram is bimodal, so a **basic automatic threshold**
   (Otsu's method — the classic image-binarisation rule) splits the two modes with no
   labels.

In [ ]:
def otsu_threshold(values, bins=64):
    """Classic Otsu threshold: the cut that maximizes between-class variance."""
    hist, edges = np.histogram(values, bins=bins)
    centers = 0.5*(edges[:-1] + edges[1:])
    total = hist.sum(); sum_all = (hist*centers).sum()
    wB = sumB = 0.0; best_var = -1.0; thr = centers[0]
    for i in range(len(hist)):
        wB += hist[i]
        if wB == 0: continue
        wF = total - wB
        if wF == 0: break
        sumB += hist[i]*centers[i]
        mB = sumB/wB; mF = (sum_all - sumB)/wF
        between = wB*wF*(mB - mF)**2
        if between > best_var: best_var, thr = between, centers[i]
    return thr

# centroid of the genuine majority (trimmed mean), then cosine distance
centroid = X.mean(0); centroid /= np.linalg.norm(centroid)
for _ in range(5):                                   # drop the farthest 30%, recompute
    d = 1.0 - X @ centroid
    centroid = X[d <= np.quantile(d, 0.70)].mean(0); centroid /= np.linalg.norm(centroid)

dist = 1.0 - X @ centroid                             # cosine distance to centroid = anomaly score
thr  = otsu_threshold(dist)                           # basic automatic threshold
print(f"cosine distance range [{dist.min():.3f}, {dist.max():.3f}]")
print(f"Otsu threshold = {thr:.3f}")

## 10. Annotated, playable video (boxes + keypoints + verdict)

Right under the detection, render the result back onto the video. Every frame gets the
**face box**, the **5 facial keypoints** (eyes, nose, mouth corners — the points SCRFD
uses to align the face for the encoder), and the **verdict** (green = genuine, red =
flagged fake, with the score). Saved as an H.264 `.mp4` you can play.

In [ ]:
# per-frame anomaly score / flag aligned to original frame positions
score_full = np.full(len(manip), np.nan)
score_full[pos] = dist
flag_full = score_full > thr            # NaN > thr -> False (frames without a face)

def annotate(frame, finfo, score, flagged):
    img = frame.copy()
    if finfo is not None:
        bbox, kps = finfo
        x1, y1, x2, y2 = bbox.astype(int)
        color = (0, 0, 255) if flagged else (0, 200, 0)      # BGR: red if flagged
        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
        for kx, ky in kps.astype(int):
            cv2.circle(img, (int(kx), int(ky)), 2, (0, 255, 255), -1)   # yellow keypoints
        label = (f"FAKE? {score:.2f}" if flagged else f"ok {score:.2f}") if score==score else "no face"
        cv2.putText(img, label, (x1, max(14, y1-6)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    return img

annotated = [annotate(manip[i], info[i], score_full[i], bool(flag_full[i])) for i in range(len(manip))]
anno_path = write_video(CONFIG["out_dir"]/"manipulated_annotated.mp4", annotated, fps=CONFIG["fps"])

# inline sample frames + player
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, j in zip(axes, [lo//2, (lo+hi)//2, min(len(manip)-1, hi+ (len(manip)-hi)//2)]):
    show_bgr(annotated[j], f"frame #{j}", ax)
plt.tight_layout(); plt.show()
display(Video(str(anno_path), embed=False, width=480))
print("Playable videos in ./outputs/ :  manipulated.mp4  +  manipulated_annotated.mp4")

## 11. Evaluation

Per-frame ground truth (`y`) vs the cosine-distance score. Swapped frames are the
minority, so we report **ROC-AUC** and **average precision (AP)** (threshold-free), plus
precision/recall **at the Otsu threshold**.

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

auc = roc_auc_score(y, dist)
ap  = average_precision_score(y, dist)
pred = dist > thr
tp = int((pred & (y==1)).sum()); fp = int((pred & (y==0)).sum())
fn = int((~pred & (y==1)).sum()); tn = int((~pred & (y==0)).sum())
prec = tp/(tp+fp) if tp+fp else 0.0
rec  = tp/(tp+fn) if tp+fn else 0.0

print(f"frames scored: {len(y)} | fake: {int(y.sum())} ({y.mean():.0%})")
print(f"ROC-AUC = {auc:.3f}")
print(f"AP      = {ap:.3f}")
print(f"at Otsu threshold {thr:.3f}:  precision = {prec:.2%}  recall = {rec:.2%}")
print(f"confusion:  TP={tp}  FP={fp}  FN={fn}  TN={tn}")

## 12. Visualization

The key plot: a **histogram of cosine distances from the centroid, coloured by the
true label** (green = real, red = fake). A clean bimodal split with the Otsu threshold
in the valley is exactly what we want to see. We then show the score over time and a
montage of the most anomalous frames.

In [ ]:
# === histogram of cosine distances to the centroid, coloured by fake/real label ===
plt.figure(figsize=(9, 5))
bins = np.linspace(dist.min(), dist.max(), 40)
plt.hist(dist[y==0], bins=bins, alpha=0.7, color="tab:green",
         label=f"real  (n={int((y==0).sum())})")
plt.hist(dist[y==1], bins=bins, alpha=0.7, color="tab:red",
         label=f"fake  (n={int((y==1).sum())})")
plt.axvline(thr, ls="--", color="k", lw=2, label=f"Otsu threshold = {thr:.3f}")
plt.xlabel("cosine distance to centroid  (1 - cos)")
plt.ylabel("number of frames")
plt.title("Distance-to-centroid histogram, coloured by label")
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# score over time, with the true swapped segment shaded
plt.figure(figsize=(12, 4))
swap_pos = pos[y==1]
if len(swap_pos):
    plt.axvspan(swap_pos.min(), swap_pos.max(), color="red", alpha=0.12, label="true swapped segment")
plt.plot(pos[y==0], dist[y==0], ".", color="tab:green", label="real")
plt.plot(pos[y==1], dist[y==1], ".", color="tab:red",   label="fake")
plt.axhline(thr, ls="--", color="k", label="threshold")
plt.xlabel("frame index"); plt.ylabel("cosine distance to centroid")
plt.title("Anomaly score over time"); plt.legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# montage of the most anomalous frames (OK = truly swapped)
order = np.argsort(-dist)[:6]
fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for ax, o in zip(axes.ravel(), order):
    tag = "OK" if y[o]==1 else "x"
    show_bgr(manip[pos[o]], f"#{pos[o]}  dist={dist[o]:.2f}  [{tag}]", ax)
fig.suptitle("Most anomalous frames (OK = truly swapped)", y=1.02)
plt.tight_layout(); plt.show()

## 13. Conclusion, limitations, future work

**What we showed.** Swapping an identity into part of a video moves those frames to a
different region of the ArcFace latent space. Because the tampered frames are a sparse
minority, a robust unsupervised outlier detector — fit on all frames, no clean
reference needed — flags them with high ROC-AUC / AP and **localizes the segment in
time**, and we can render the verdict back onto the video.

**Limitations (honestly).**
- This assumes the fakes are a **minority** and the genuine identity is **consistent**.
  If most of the video is swapped, "normal" is no longer the genuine cluster.
- ArcFace is an *identity* space: it detects that the **identity changed**. A
  manipulation preserving identity (re-enactment, expression transfer) won't show up
  here — use artifact detectors or temporal-texture cues for those.
- A single hard cut between genuine and swapped is the easy case; real deepfakes blend more smoothly.

**Where to go next.**
- **Temporal smoothing**: average the score over a sliding window to recover contiguous segments.
- **Reference identity**: if you have a verified photo of person *A*, score distance to *that* embedding.
- **Multiple faces / tracking**: score each tracked face separately.
- Compare against an **autoencoder reconstruction error** over face crops.
